# explainer — Classification Example

This notebook demonstrates every feature of the `explainer` package using a **binary classification** task.

**Dataset:** Wisconsin Breast Cancer (sklearn built-in)  
**Model:** XGBoost `XGBClassifier`  
**Features covered:**
- Basic SHAP explanations (aggregate + per-observation)
- All 10 visualisation types
- Interaction effects
- Bootstrap confidence intervals
- Feature groups
- Saving plots to disk
- Multiclass extension (Iris with Random Forest)

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────
import warnings
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.datasets import load_breast_cancer, load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

from transpyrancy import Explainer

warnings.filterwarnings('ignore')
print('explainer ready ✓')

---
## 1 · Load Data & Train Model

In [ ]:
X, y = load_breast_cancer(return_X_y=True, as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    random_state=42,
    eval_metric='logloss',
)
model.fit(X_train, y_train)

accuracy = (model.predict(X_test) == y_test).mean()
print(f'Test accuracy: {accuracy:.3f}')
print(f'Training rows : {len(X_train)}')
print(f'Test rows     : {len(X_test)}')
print(f'Features      : {X_train.shape[1]}')

---
## 2 · Construct the Explainer

In [ ]:
# Define feature groups for later use
feature_groups = {
    'size':     ['mean radius', 'mean area', 'mean perimeter',
                 'radius error', 'area error', 'perimeter error',
                 'worst radius', 'worst area', 'worst perimeter'],
    'texture':  ['mean texture', 'texture error', 'worst texture'],
    'shape':    ['mean smoothness', 'mean compactness', 'mean concavity',
                 'mean concave points', 'mean symmetry', 'mean fractal dimension',
                 'smoothness error', 'compactness error', 'concavity error',
                 'concave points error', 'symmetry error', 'fractal dimension error',
                 'worst smoothness', 'worst compactness', 'worst concavity',
                 'worst concave points', 'worst symmetry', 'worst fractal dimension'],
}

exp = Explainer(
    model_type='xgb',
    model=model,
    background_data=X_train,   # recommended for TreeExplainer
    feature_groups=feature_groups,
)
print(exp)

---
## 3 · Explain the Test Set

In [ ]:
result = exp.explain(
    X_test,
    compute_interactions=True,
    compute_confidence_intervals=True,
    n_bootstrap=50,
    ci_level=0.95,
)

# Rich text summary
print(result)

---
## 4 · Inspect the Data

In [ ]:
# Per-observation SHAP values — shape (n_obs, n_features)
print('SHAP values shape:', result.shap_values.shape)
result.shap_values.head(3)

In [ ]:
# Global feature importance — mean |SHAP| per feature, sorted
print('Global feature importance:')
result.aggregate

In [ ]:
# Top-10 features
print('Top 10 features:')
result.top_features(10)

In [ ]:
# Model predictions
print('Predictions (first 10):', result.predictions[:10])
print('Proba (first 3 rows):')
pd.DataFrame(result.prediction_proba[:3], columns=['P(benign)', 'P(malignant)'])

In [ ]:
# Model baseline
print(f'Expected value (baseline): {result.expected_value:.4f}')

---
## 4.5 · Built-in Help System

`result.help()` renders plain-English explanations for every attribute and plot —
context-aware (it knows which optional outputs you enabled) and Jupyter-formatted.

In [ ]:
# Full overview — shows what's available, what's missing, and how to enable it.
# In Jupyter this renders as formatted Markdown.
result.help()

In [ ]:
# Zoom in on a specific attribute
result.help('shap_values')

In [ ]:
# Get guidance on how to read a plot before generating it
result.help('plot_waterfall')

In [ ]:
# Aliases work too — short names for common topics
result.help('ci')          # confidence intervals
# result.help('interactions')
# result.help('groups')
# result.help('expected_value')

In [ ]:
# Before calling explain(), you can get a library overview from the class
from transpyrancy import Explainer
Explainer.help()

---
## 5 · Visualisations

### 5a · Summary (Beeswarm)

In [ ]:
# Distribution of SHAP values per feature across all test observations
# Colour = feature value (red = high, blue = low)
result.plot_summary(max_display=20)

### 5b · Feature Importance Bar Chart

In [ ]:
result.plot_feature_importance(top_n=15)

### 5c · Waterfall — Single Observation

Pick the first misclassified observation and explain it.

In [ ]:
# Find a misclassified observation
misclassified = np.where(result.predictions != y_test.values)[0]
if len(misclassified) > 0:
    idx = int(misclassified[0])
    print(f'Misclassified observation at index {idx}')
    print(f'  True label       : {y_test.values[idx]}')
    print(f'  Predicted label  : {result.predictions[idx]}')
    print(f'  Predicted proba  : {result.prediction_proba[idx]}')
else:
    idx = 0
    print('All correctly classified — showing index 0.')

result.plot_waterfall(index=idx, max_display=15)

### 5d · Force Plot — Single Observation

In [ ]:
result.plot_force(index=idx)

### 5e · Decision Plot — All Observations

Each line = one observation's cumulative path from baseline → prediction.

In [ ]:
result.plot_decision(max_display=15)

### 5f · SHAP Heatmap

In [ ]:
result.plot_heatmap(max_display=15)

### 5g · Dependence Plot

In [ ]:
# How does 'worst radius' SHAP value change with its raw value?
# Colour = the auto-detected best interaction feature
result.plot_dependence('worst radius')

In [ ]:
# Specify interaction feature manually
result.plot_dependence('worst radius', interaction_feature='worst concave points')

### 5h · Feature Group Contributions

In [ ]:
# Which semantic group matters most?
print('Group contributions (first 5 rows):')
print(result.group_contributions.head())
result.plot_group_contributions()

### 5i · Confidence Intervals

In [ ]:
# Aggregate CIs — mean SHAP ± 95% bootstrap CI per feature
result.plot_confidence_intervals(top_n=12)

In [ ]:
# CI for a single (misclassified) observation
result.plot_confidence_intervals(top_n=12, observation_index=idx)

### 5j · Interaction Effects

In [ ]:
print('Interaction effects shape:', result.interaction_effects.shape)
# Top features that interact with 'worst radius'
result.plot_interaction_heatmap('worst radius', top_n=10)

---
## 6 · Save All Plots to Disk

In [ ]:
result_saved = exp.explain(
    X_test,
    visualize=False,          # do not show inline
    save_plots='./plots/classification/',
)

import os
files = sorted(os.listdir('./plots/classification/'))
print(f'{len(files)} PNGs saved:')
for f in files:
    print(' ', f)

---
## 7 · Explaining a Single Row

In [ ]:
# Pass a 1-D numpy array (a single patient)
single_patient = X_test.values[idx]  # 1-D
single_result = exp.explain(single_patient)

print(f'n_observations: {single_result.n_observations}')
print(f'Prediction    : {single_result.predictions[0]}')
print('SHAP values   :')
single_result.shap_values.T.rename(columns={0: 'SHAP'}).sort_values('SHAP', key=abs, ascending=False).head(8)

---
## 8 · Multiclass Classification (Iris)

The API is identical — `is_multiclass=True` and `shap_values` becomes a list.

In [ ]:
Xi, yi = load_iris(return_X_y=True, as_frame=True)
Xtr, Xte, ytr, yte = train_test_split(Xi, yi, test_size=0.2, random_state=42)

iris_model = RandomForestClassifier(n_estimators=100, random_state=42)
iris_model.fit(Xtr, ytr)

iris_exp = Explainer('rf', iris_model)
iris_result = iris_exp.explain(Xte)

print(iris_result)

In [ ]:
# SHAP values is a list of 3 DataFrames
print('is_multiclass :', iris_result.is_multiclass)
print('n_classes     :', iris_result.n_classes)
print('len(shap_values):', len(iris_result.shap_values))

# Get SHAP values for class 2
iris_result.shap_values_for_class(2).head(3)

In [ ]:
# Summary plot for class 0
iris_result.plot_summary(class_index=0, max_display=4)

In [ ]:
# Feature importance aggregated across all classes
iris_result.plot_feature_importance()

In [ ]:
# Waterfall for obs 0, class 1
iris_result.plot_waterfall(index=0, class_index=1)

---
## 9 · Explainer Override (with Warning)

In [ ]:
import warnings

# Forcing 'linear' explainer on an XGBoost model triggers a UserWarning
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')
    exp_override = Explainer(
        'xgb', model,
        background_data=X_train,
        explainer_override='kernel',
    )

for w in caught:
    print('⚠ ', w.category.__name__, ':', w.message)